# Low-dimensional Q70 size-adjusted power experiments (v2)

This notebook follows the supplied `run_power_size_adjusted_v2-Copy1_multi_gpu(1).ipynb` driver. Its multi-GPU seed sharding, H0 calibration, empirical cutoffs, H1 size-adjusted power, tables, figures, and result-saving workflow are retained.

The experiment-specific changes are:

1. register the low-dimensional **Q70 DGP** (`dz=10`);
2. separately tune the two neural-network learning configurations;
3. replace multi-quantile alignment by **Q70-only alignment** with `taus=(0.70,)`;
4. use the previously selected alignment weight **lambda = 0.05**.

Both H0 and each H1 dependence point use exactly **100 Monte Carlo repetitions** at `n=400`.

Keep this notebook and the modified `ci_test.py` (the version with `size_adjusted_cutoffs` and `null_pvalues`) in the same folder. Restart the kernel after replacing `ci_test.py`.


## 0. Setup - import and verify the modified module

This cell deliberately checks the size-adjusted-power API. If it fails, Jupyter is still importing an older `ci_test.py`.

**Multi-GPU:** Section 0 retains the supplied wrapper and shards Monte Carlo seeds across `GPU_IDS = [1, 2, 3]`. GPU 0 remains skipped by default.


In [ ]:
%pip install matplotlib pandas torch

In [2]:
from copy import deepcopy
import importlib
import inspect
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

import ci_test as C
C = importlib.reload(C)

assert hasattr(C, 'size_adjusted_cutoffs'), (
    'The imported ci_test.py is old: size_adjusted_cutoffs is missing.'
)
assert 'null_pvalues' in inspect.signature(C.run_experiment).parameters, (
    'The imported ci_test.py is old: run_experiment has no null_pvalues argument.'
)

print('ci_test loaded from:', Path(C.__file__).resolve())
print('Size-adjusted API check: PASSED')
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())


# =============================================================================
# Multi-GPU: run Monte-Carlo replicates on GPUs 1/2/3 in parallel
# (GPU 0 is skipped by default because it is often full.)
# This wraps C.run_experiment; later cells keep calling C.run_experiment(...).
# =============================================================================
from joblib import Parallel, delayed

GPU_IDS = [0, 1, 2, 3]  # physical device indices to use


def _split_seeds(n_rep, n_workers):
    seeds = list(range(int(n_rep)))
    if n_workers <= 0:
        return [seeds]
    base, rem = divmod(len(seeds), n_workers)
    chunks, start = [], 0
    for i in range(n_workers):
        sz = base + (1 if i < rem else 0)
        chunks.append(seeds[start:start + sz])
        start += sz
    return chunks


def _register_q70_for_worker(ci_local):
    """Register the Q70 DGP after this worker has been pinned to one GPU."""
    torch_local = ci_local.torch

    def levels(Z):
        hx = 0.8 + 0.7 * torch_local.sigmoid(
            0.8 * Z[:, [0]] - 0.6 * Z[:, [1]] + 0.4 * torch_local.sin(Z[:, [2]])
        )
        hy = 0.8 + 0.7 * torch_local.sigmoid(
            -0.7 * Z[:, [0]] + 0.5 * Z[:, [3]] + 0.3 * torch_local.cos(Z[:, [4]])
        )
        return hx, hy

    def inverse_cdf(u, h):
        c = 16.0 * h - 8.8
        left_tail = -c + (u / 0.05) * (c - 1.0)
        center_left = -1.0 + (u - 0.05) / 0.45
        center_right = h * (u - 0.50) / 0.20
        upper_tail = h + 0.2 * (u - 0.70) / 0.30
        return torch_local.where(
            u < 0.05,
            left_tail,
            torch_local.where(
                u < 0.50,
                center_left,
                torch_local.where(u < 0.70, center_right, upper_tail),
            ),
        )

    def checkerboard(n, tau, rho, device):
        delta = rho * tau * (1.0 - tau)
        probabilities = torch_local.tensor(
            [
                tau * tau + delta,
                tau * (1.0 - tau) - delta,
                tau * (1.0 - tau) - delta,
                (1.0 - tau) ** 2 + delta,
            ],
            dtype=torch_local.float32,
            device=device,
        )
        cell = torch_local.multinomial(probabilities, n, replacement=True)
        x_high = (cell >= 2).reshape(-1, 1)
        y_high = ((cell == 1) | (cell == 3)).reshape(-1, 1)
        ux = torch_local.where(
            x_high,
            tau + (1.0 - tau) * torch_local.rand(n, 1, device=device),
            tau * torch_local.rand(n, 1, device=device),
        )
        uy = torch_local.where(
            y_high,
            tau + (1.0 - tau) * torch_local.rand(n, 1, device=device),
            tau * torch_local.rand(n, 1, device=device),
        )
        return ux, uy

    def sample(n, hypothesis='H0', device=None, dz=10, alpha_x=0.10, **_):
        device = device or ci_local.get_device(prefer_gpu=True)
        Z = torch_local.randn(n, int(dz), device=device)
        hx, hy = levels(Z)
        if hypothesis.upper() == 'H0':
            ux = torch_local.rand(n, 1, device=device)
            uy = torch_local.rand(n, 1, device=device)
        elif hypothesis.upper() == 'H1':
            ux, uy = checkerboard(n, 0.70, float(alpha_x), device)
        else:
            raise ValueError("hypothesis must be 'H0' or 'H1'.")
        return inverse_cdf(ux, hx), inverse_cdf(uy, hy), Z

    def oracle(Z, m, device=None, **_):
        device = device or Z.device
        Z = Z.to(device)
        hx, hy = levels(Z)
        ux = torch_local.rand(Z.shape[0], m, 1, device=device)
        uy = torch_local.rand(Z.shape[0], m, 1, device=device)
        return inverse_cdf(ux, hx.unsqueeze(1)), inverse_cdf(uy, hy.unsqueeze(1))

    ci_local.register_dgp(ci_local.DGP(
        name='q70_lowdim',
        sample=sample,
        oracle=oracle,
        description='Worker-local copy of the low-dimensional Q70 DGP.',
    ))


def _multi_gpu_chunk_worker(gpu_id, seeds, params):
    """One loky worker: pin a physical GPU, then run its seed chunk."""
    import os
    os.environ['CUDA_VISIBLE_DEVICES'] = str(int(gpu_id))

    import numpy as np
    import ci_test as ci_local

    # Register Q70 only after CUDA_VISIBLE_DEVICES has been applied.
    if params.get('register_q70', False):
        _register_q70_for_worker(ci_local)

    pvals = []
    for s in seeds:
        pvals.append(
            ci_local._one_replicate(
                int(s),
                params['n'],
                params['hypothesis'],
                params['config'],
                params['oracle'],
                params['data_kwargs'],
                True,
                params['dgp'],
            )
        )
    return list(seeds), np.asarray(pvals, dtype=float)


_ORIG_RUN_EXPERIMENT = C.run_experiment


def run_experiment(
    n=400,
    hypothesis='H0',
    n_rep=200,
    config=None,
    oracle=False,
    levels=(0.10, 0.05),
    data_kwargs=None,
    dgp='skew',
    n_jobs=1,
    prefer_gpu=True,
    verbose=True,
    null_pvalues=None,
    gpu_ids=None,
    **kwargs,
):
    """Drop-in wrapper around C.run_experiment with multi-GPU replicate sharding."""
    data_kwargs = dict(data_kwargs or {})
    levels = tuple(float(x) for x in levels)
    ids = [int(g) for g in (GPU_IDS if gpu_ids is None else gpu_ids)]

    use_multi = bool(prefer_gpu) and torch.cuda.is_available() and len(ids) >= 1
    if not use_multi:
        return _ORIG_RUN_EXPERIMENT(
            n=n,
            hypothesis=hypothesis,
            n_rep=n_rep,
            config=config,
            oracle=oracle,
            levels=levels,
            data_kwargs=data_kwargs,
            dgp=dgp,
            n_jobs=n_jobs,
            prefer_gpu=prefer_gpu,
            verbose=verbose,
            null_pvalues=null_pvalues,
            **kwargs,
        )

    chunks = _split_seeds(n_rep, len(ids))
    params = dict(
        n=n,
        hypothesis=hypothesis,
        config=config,
        oracle=oracle,
        data_kwargs=data_kwargs,
        dgp=dgp,
        register_q70=(dgp == 'q70_lowdim'),
    )
    jobs = [(gid, chunk) for gid, chunk in zip(ids, chunks) if chunk]

    if verbose:
        print(
            f"[info] multi-GPU on devices {[g for g, _ in jobs]} | reps={n_rep} | "
            f"split={[len(c) for _, c in jobs]}"
        )

    parts = Parallel(n_jobs=len(jobs), backend='loky', verbose=0)(
        delayed(_multi_gpu_chunk_worker)(gid, chunk, params) for gid, chunk in jobs
    )

    seed_to_pval = {}
    for seed_list, arr in parts:
        for s, p in zip(seed_list, arr):
            seed_to_pval[int(s)] = float(p)

    pvals = np.asarray([seed_to_pval[s] for s in range(int(n_rep))], dtype=float)
    rej = {lvl: float(np.mean(pvals < lvl)) for lvl in levels}

    if verbose:
        gtag = C.get_dgp(dgp).name
        mtag = (
            'ORACLE'
            if oracle
            else f"depth={(config or {}).get('depth', C.DEFAULT_CONFIG['depth'])}"
        )
        print(
            f"[{hypothesis} dgp={gtag} {mtag} n={n} reps={n_rep}] "
            + '  '.join(f'rej@{lvl:.2f}={rej[lvl]:.3f}' for lvl in levels)
        )

    result = dict(rejection=rej, pvalues=pvals)
    if null_pvalues is not None:
        cutoffs = C.size_adjusted_cutoffs(null_pvalues, levels=levels)
        cutoffs = {float(k): float(v) for k, v in dict(cutoffs).items()}
        result['cutoffs'] = cutoffs
        result['size_adjusted_power'] = {
            lvl: float(np.mean(pvals <= cutoffs[lvl])) for lvl in levels
        }
    return result


C.run_experiment = run_experiment

print('Multi-GPU wrapper enabled. GPU_IDS =', GPU_IDS)
if torch.cuda.is_available():
    print('Visible CUDA device count (this process):', torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        try:
            free, total = torch.cuda.mem_get_info(i)
            print(
                f'  cuda:{i}: free={free/1024**3:.2f} GiB / total={total/1024**3:.2f} GiB'
            )
        except Exception as e:
            print(f'  cuda:{i}: mem_get_info unavailable ({e})')



ci_test loaded from: /home/23099458d/projects/ci_new/ci_test.py
Size-adjusted API check: PASSED
PyTorch: 2.8.0+cu128
CUDA available: True
Multi-GPU wrapper enabled. GPU_IDS = [0, 1, 2, 3]
Visible CUDA device count (this process): 4
  cuda:0: free=38.65 GiB / total=39.49 GiB
  cuda:1: free=38.65 GiB / total=39.49 GiB
  cuda:2: free=38.65 GiB / total=39.49 GiB
  cuda:3: free=38.65 GiB / total=39.49 GiB


## 0.1 Register the low-dimensional Q70 DGP

For each observation, `Z` is 10-dimensional standard Gaussian. The nonlinear functions `h_X(Z)` and `h_Y(Z)` determine the conditional 70th percentiles of `X` and `Y`.

The piecewise inverse CDF is constructed so that, conditional on `Z`, the mean and median remain zero while `Q70(X|Z)=h_X(Z)` and `Q70(Y|Z)=h_Y(Z)`. Under H0, the latent uniforms are independent. Under H1, a checkerboard copula couples only the two Q70 exceedance indicators while preserving both Uniform(0,1) marginals.


In [3]:
def _q70_levels(Z):
    if Z.ndim != 2 or Z.shape[1] < 5:
        raise ValueError('The Q70 DGP requires dz >= 5.')
    hx = 0.8 + 0.7 * torch.sigmoid(
        0.8 * Z[:, [0]] - 0.6 * Z[:, [1]] + 0.4 * torch.sin(Z[:, [2]])
    )
    hy = 0.8 + 0.7 * torch.sigmoid(
        -0.7 * Z[:, [0]] + 0.5 * Z[:, [3]] + 0.3 * torch.cos(Z[:, [4]])
    )
    return hx, hy


def _q70_inverse_cdf(u, h):
    """Piecewise quantile function with mean=0, median=0 and Q(0.70)=h."""
    c = 16.0 * h - 8.8
    left_tail = -c + (u / 0.05) * (c - 1.0)
    center_left = -1.0 + (u - 0.05) / 0.45
    center_right = h * (u - 0.50) / 0.20
    upper_tail = h + 0.2 * (u - 0.70) / 0.30
    return torch.where(
        u < 0.05,
        left_tail,
        torch.where(
            u < 0.50,
            center_left,
            torch.where(u < 0.70, center_right, upper_tail),
        ),
    )


def _q70_checkerboard_uniforms(n, tau, rho, device):
    """Uniform marginals whose Q70-cell indicators have positive dependence."""
    if not 0.0 <= rho <= 1.0:
        raise ValueError('alpha_x must lie in [0, 1].')

    delta = rho * tau * (1.0 - tau)
    cell_probabilities = torch.tensor(
        [
            tau * tau + delta,
            tau * (1.0 - tau) - delta,
            tau * (1.0 - tau) - delta,
            (1.0 - tau) ** 2 + delta,
        ],
        dtype=torch.float32,
        device=device,
    )
    cell = torch.multinomial(cell_probabilities, n, replacement=True)

    x_high = (cell >= 2).reshape(-1, 1)
    y_high = ((cell == 1) | (cell == 3)).reshape(-1, 1)

    ux = torch.where(
        x_high,
        tau + (1.0 - tau) * torch.rand(n, 1, device=device),
        tau * torch.rand(n, 1, device=device),
    )
    uy = torch.where(
        y_high,
        tau + (1.0 - tau) * torch.rand(n, 1, device=device),
        tau * torch.rand(n, 1, device=device),
    )
    return ux, uy


def sample_q70_lowdim(
    n,
    hypothesis='H0',
    device=None,
    dz=10,
    alpha_x=0.10,
    **_,
):
    device = device or C.get_device(prefer_gpu=True)
    if int(dz) < 5:
        raise ValueError('Use dz >= 5 for the Q70 DGP.')

    Z = torch.randn(n, int(dz), device=device)
    hx, hy = _q70_levels(Z)

    if hypothesis.upper() == 'H0':
        ux = torch.rand(n, 1, device=device)
        uy = torch.rand(n, 1, device=device)
    elif hypothesis.upper() == 'H1':
        ux, uy = _q70_checkerboard_uniforms(
            n=n,
            tau=0.70,
            rho=float(alpha_x),
            device=device,
        )
    else:
        raise ValueError("hypothesis must be 'H0' or 'H1'.")

    X = _q70_inverse_cdf(ux, hx)
    Y = _q70_inverse_cdf(uy, hy)
    return X, Y, Z


def oracle_q70_lowdim(Z, m, device=None, **_):
    """Independent samples from the exact conditional marginals."""
    device = device or Z.device
    Z = Z.to(device)
    hx, hy = _q70_levels(Z)
    hx = hx.unsqueeze(1)
    hy = hy.unsqueeze(1)
    ux = torch.rand(Z.shape[0], m, 1, device=device)
    uy = torch.rand(Z.shape[0], m, 1, device=device)
    return _q70_inverse_cdf(ux, hx), _q70_inverse_cdf(uy, hy)


Q70_DGP = C.register_dgp(C.DGP(
    name='q70_lowdim',
    sample=sample_q70_lowdim,
    oracle=oracle_q70_lowdim,
    description=(
        'Low-dimensional Q70 DGP: dz=10; conditional means and medians are zero; '
        'Q0.70 varies nonlinearly with Z; H1 couples the Q70 exceedance cells. '
        'Knobs: dz, alpha_x.'
    ),
))

# Small shape check only; the 100-repetition experiments start below.
_X, _Y, _Z = Q70_DGP.sample(
    16,
    hypothesis='H0',
    device=torch.device('cpu'),
    dz=10,
)
assert _X.shape == _Y.shape == (16, 1)
assert _Z.shape == (16, 10)
print('Registered:', Q70_DGP.name)
print(Q70_DGP.description)


Registered: q70_lowdim
Low-dimensional Q70 DGP: dz=10; conditional means and medians are zero; Q0.70 varies nonlinearly with Z; H1 couples the Q70 exceedance cells. Knobs: dz, alpha_x.


## 1. Global experiment settings

Use `RUN_PROFILE='quick'` for an initial check. For final tables, use more H0 replications because the empirical cutoffs themselves must be estimated accurately.


In [9]:
RUN_PROFILE = 'final'      # 'quick' or 'final'

if RUN_PROFILE == 'quick':
    N_REP_ORACLE = 50
    N_REP_H0 = 100
    N_REP_H1 = 100
else:
    N_REP_ORACLE = 100
    N_REP_H0 = 100
    N_REP_H1 = 100

N = 400
LEVELS = (0.10, 0.05)
DGP_NAME = 'q70_lowdim'
N_JOBS = -1
PREFER_GPU = True

# DGP arguments shared by H0 and H1. Leave empty to use the built-in skew defaults.
BASE_DATA_KWARGS = {'dz': 10}

# Dependence strengths under H1.
ALPHA_GRID = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40]


# Multi-GPU devices used by the Section 0 wrapper (skip busy GPU 0).
GPU_IDS = [0, 1, 2, 3]
print('Using GPU_IDS =', GPU_IDS)

print({
    'profile': RUN_PROFILE,
    'n': N,
    'levels': LEVELS,
    'H0 replications': N_REP_H0,
    'H1 replications per alpha_x': N_REP_H1,
    'DGP': DGP_NAME,
})


Using GPU_IDS = [0, 1, 2, 3]
{'profile': 'final', 'n': 400, 'levels': (0.1, 0.05), 'H0 replications': 100, 'H1 replications per alpha_x': 100, 'DGP': 'q70_lowdim'}


## 2. Inspect and choose the DGP

The same DGP, sample size, standardization, bootstrap settings, and non-H1 data arguments must be used in the H0 calibration and H1 power stages.


In [10]:
for name, dgp in C.DGPS.items():
    print(f'{name:16s}: {dgp.description}')

assert DGP_NAME in C.DGPS, f'Unknown DGP_NAME={DGP_NAME!r}'


skew            : This project's hard case: skewed lognormal noise, heteroscedastic spread, nonlinear means. Knobs: dx, dy, dz, nstd, dist_z, alpha_x.
gaussian        : Zhang et al. Section 4.1 (Sim 4/5): Z=e3, Y=Z+e1, X=Z+d*e1+(1-d)*e2, d~Bernoulli(alpha_x). Gaussian / homoscedastic / linear. Knobs: dz, alpha_x.
skew_linear     : Simulation 8 (Sec 1.3): the skewed DGP but with LINEAR means m_X=0.8z, m_Y=-0.6z. Knobs: dx, dy, dz, nstd, dist_z, alpha_x.
heteroskedastic : Simulation 5 (Sec 3.1): Y=Z+ey, X=sigma(Z)*ex, sigma(Z)=0.3+1.2|Z|. Gaussian but heteroscedastic with ZERO conditional mean. Knobs: dz, alpha_x.
student_t       : Simulation 5 (Sec 4): heavy-tailed, mX=mY=Z, s(Z)=0.5+|Z|, standardized Student-t noise. Knobs: dz, alpha_x, df (default 3).
q70_lowdim      : Low-dimensional Q70 DGP: dz=10; conditional means and medians are zero; Q0.70 varies nonlinearly with Z; H1 couples the Q70 exceedance cells. Knobs: dz, alpha_x.


## 3. Define the two methods

Two designs remain available exactly as in the supplied driver:

- `separately_tuned`: compare the two Q70-specific neural-network learning configurations;
- `strict_ablation`: keep every learning setting identical and change only `align_mode` and `lambda_align`.

The requested default is `separately_tuned`. Q70 alignment uses `taus=(0.70,)` and the previously selected `lambda_align=0.05`.


In [11]:
COMPARISON_MODE = 'separately_tuned'   # 'separately_tuned' or 'strict_ablation'

# -----------------------------------------------------------------------------
# Q70 tuned configuration: no alignment (lambda=0)
# -----------------------------------------------------------------------------
CONFIG_L0_TUNED = dict(C.DEFAULT_CONFIG)
CONFIG_L0_TUNED.update(
    depth=3,
    width=1024,
    noise_dim=8,
    dropout=0.0,

    lr=1.0e-3,
    epochs=1000,
    batch_size=128,
    grad_clip=None,
    weight_decay=1e-5,
    M_train=40,
    mmd_w_laplacian=1.0,
    mmd_w_gaussian=1.0,

    early_stop=True,
    min_epochs=140,
    patience=110,
    min_delta=2e-5,
    lr_scheduler=True,
    lr_factor=0.3,
    lr_patience=20,
    min_lr_frac=0.02,

    align_mode='none',
    lambda_align=0.0,
    taus=(0.70,),
    align_samples=64,

    n_folds=2,
    M_test=100,
    n_boot=1000,
    boot_rv='gaussian',
    standardize=True,
)

# -----------------------------------------------------------------------------
# Q70 tuned configuration: Q70-only alignment (lambda=0.05)
# -----------------------------------------------------------------------------
CONFIG_L005_TUNED = dict(C.DEFAULT_CONFIG)
CONFIG_L005_TUNED.update(
    depth=3,
    width=1024,
    noise_dim=8,
    dropout=0.0,

    lr=7.5e-4,
    epochs=1100,
    batch_size=128,
    grad_clip=None,
    weight_decay=1e-5,
    M_train=30,
    mmd_w_laplacian=1.0,
    mmd_w_gaussian=1.0,

    early_stop=True,
    min_epochs=160,
    patience=130,
    min_delta=2e-5,
    lr_scheduler=True,
    lr_factor=0.3,
    lr_patience=20,
    min_lr_frac=0.02,

    align_mode='quantile',
    lambda_align=0.05,
    taus=(0.70,),
    align_samples=128,

    n_folds=2,
    M_test=100,
    n_boot=1000,
    boot_rv='gaussian',
    standardize=True,
)

# Strict ablation option: both methods share the no-alignment learning settings;
# only align_mode and lambda_align differ. The inactive Q70 tau and sample count
# are already present in STRICT_BASE, so the assertion below remains exact.
STRICT_BASE = deepcopy(CONFIG_L0_TUNED)

CONFIG_L0_STRICT = deepcopy(STRICT_BASE)
CONFIG_L0_STRICT.update(align_mode='none', lambda_align=0.0)

CONFIG_L005_STRICT = deepcopy(STRICT_BASE)
CONFIG_L005_STRICT.update(
    align_mode='quantile',
    lambda_align=0.05,
)

if COMPARISON_MODE == 'separately_tuned':
    METHODS = {
        'lambda_0': CONFIG_L0_TUNED,
        'lambda_005': CONFIG_L005_TUNED,
    }
elif COMPARISON_MODE == 'strict_ablation':
    METHODS = {
        'lambda_0': CONFIG_L0_STRICT,
        'lambda_005': CONFIG_L005_STRICT,
    }
else:
    raise ValueError("COMPARISON_MODE must be 'separately_tuned' or 'strict_ablation'.")

METHOD_LABELS = {
    'lambda_0': 'No alignment (lambda=0)',
    'lambda_005': 'Q70 alignment (tau=0.70, lambda=0.05)',
}

print('Comparison mode:', COMPARISON_MODE)
for method_id in METHODS:
    print(method_id, '->', METHOD_LABELS[method_id])


Comparison mode: separately_tuned
lambda_0 -> No alignment (lambda=0)
lambda_005 -> Q70 alignment (tau=0.70, lambda=0.05)


### 3.1 Verify which parameters differ

In `strict_ablation` mode, only `align_mode` and `lambda_align` should differ. This cell prevents accidental comparison of mislabeled configurations.


In [7]:
def config_difference_table(config_a, config_b):
    keys = sorted(set(config_a) | set(config_b))
    rows = []
    for key in keys:
        value_a = config_a.get(key, '<missing>')
        value_b = config_b.get(key, '<missing>')
        if value_a != value_b:
            rows.append({
                'parameter': key,
                'lambda_0': value_a,
                'lambda_005': value_b,
            })
    return pd.DataFrame(rows)

CONFIG_DIFFERENCES = config_difference_table(METHODS['lambda_0'], METHODS['lambda_005'])
display(CONFIG_DIFFERENCES)

if COMPARISON_MODE == 'strict_ablation':
    allowed = {'align_mode', 'lambda_align'}
    unexpected = set(CONFIG_DIFFERENCES['parameter']) - allowed
    assert not unexpected, f'Unexpected differences in strict ablation: {sorted(unexpected)}'


,parameter,lambda_0,lambda_005
0,M_train,40,30
1,align_mode,none,quantile
2,align_samples,64,128
3,epochs,1000,1100
4,lambda_align,0.0,0.05
5,lr,0.001,0.00075
6,min_epochs,140,160
7,patience,110,130


## 4. Oracle H0 sanity check

Run this before training learned generators. Oracle size should be near the nominal levels. If oracle is abnormal, inspect the statistic/bootstrap before tuning neural-network parameters.


In [ ]:
ORACLE_RESULT = C.run_experiment(
    n=N,
    hypothesis='H0',
    n_rep=N_REP_ORACLE,
    config=METHODS['lambda_0'],
    dgp=DGP_NAME,
    oracle=True,
    levels=LEVELS,
    data_kwargs=BASE_DATA_KWARGS,
    n_jobs=N_JOBS,
    prefer_gpu=PREFER_GPU,
    verbose=True,
)
ORACLE_RESULT


## 5. Run matching H0 calibration experiments

This is the required first stage for size-adjusted power. Each method receives its **own** H0 p-values obtained using the same method configuration, DGP, sample size, and non-H1 data settings.


In [12]:
H0_RESULTS = {}

for method_id, config in METHODS.items():
    print()
    print('=' * 88)
    print('H0 calibration:', METHOD_LABELS[method_id])
    print('=' * 88)

    H0_RESULTS[method_id] = C.run_experiment(
        n=N,
        hypothesis='H0',
        n_rep=N_REP_H0,
        config=config,
        dgp=DGP_NAME,
        oracle=False,
        levels=LEVELS,
        data_kwargs=BASE_DATA_KWARGS,
        n_jobs=N_JOBS,
        prefer_gpu=PREFER_GPU,
        verbose=True,
    )



H0 calibration: No alignment (lambda=0)
[info] multi-GPU on devices [0, 1, 2, 3] | reps=100 | split=[25, 25, 25, 25]
[H0 dgp=q70_lowdim depth=3 n=400 reps=100] rej@0.10=0.150  rej@0.05=0.090

H0 calibration: Q70 alignment (tau=0.70, lambda=0.05)
[info] multi-GPU on devices [0, 1, 2, 3] | reps=100 | split=[25, 25, 25, 25]
[H0 dgp=q70_lowdim depth=3 n=400 reps=100] rej@0.10=0.150  rej@0.05=0.120


## 6. Inspect empirical Type I error and adjusted cutoffs

The modified `ci_test.py` converts each method's H0 p-values into empirical level-specific cutoffs. Those cutoffs, rather than fixed 0.05 and 0.10, are used in H1.


In [ ]:
calibration_rows = []
ADJUSTED_CUTOFFS = {}

for method_id, h0_result in H0_RESULTS.items():
    p0 = np.asarray(h0_result['pvalues'])
    cutoffs = C.size_adjusted_cutoffs(h0_result, levels=LEVELS)
    ADJUSTED_CUTOFFS[method_id] = cutoffs

    for level in LEVELS:
        cutoff = cutoffs[float(level)]
        calibration_rows.append({
            'method_id': method_id,
            'method': METHOD_LABELS[method_id],
            'nominal_level': float(level),
            'raw_H0_rejection': h0_result['rejection'][level],
            'adjusted_cutoff': cutoff,
            'H0_rejection_at_adjusted_cutoff': float(np.mean(p0 <= cutoff)),
            'H0_replications': len(p0),
        })

CALIBRATION_TABLE = pd.DataFrame(calibration_rows)
display(CALIBRATION_TABLE)


### 6.1 Save H0 calibration p-values

H0 calibration is expensive. Save it before running the H1 grid. After a kernel restart, you may reload these p-values instead of repeating H0.


In [ ]:
H0_FILE = Path(f'h0_calibration_{DGP_NAME}_{COMPARISON_MODE}_v2.npz')

np.savez_compressed(
    H0_FILE,
    **{
        method_id: np.asarray(result['pvalues'])
        for method_id, result in H0_RESULTS.items()
    },
)

CALIBRATION_TABLE.to_csv(
    f'h0_calibration_summary_{DGP_NAME}_{COMPARISON_MODE}_v2.csv',
    index=False,
)

print('Saved H0 p-values to:', H0_FILE.resolve())


### 6.2 Optional: reload previously saved H0 p-values

Run this only after a kernel restart when you want to skip Section 5.


In [ ]:
from pathlib import Path
import numpy as np

# 优先检查 notebook 主目录中保存的 H0 文件
H0_FILE = Path(
    f"h0_calibration_{DGP_NAME}_{COMPARISON_MODE}_v2.npz"
)

# 如果主目录没有，则检查结果文件夹
if not H0_FILE.exists():
    H0_FILE = Path(
        f"size_adjusted_power_results_{COMPARISON_MODE}_v2"
    ) / "h0_pvalues.npz"

if not H0_FILE.exists():
    raise FileNotFoundError(
        "没有找到保存的 H0 p-values。"
        "如果之前只记录了 rej rate，而没有保存完整 p-values，"
        "则需要重新运行一次 H0 calibration。"
    )

loaded = np.load(H0_FILE)

H0_RESULTS = {}

for method_id in METHODS:
    if method_id not in loaded.files:
        raise KeyError(
            f"H0 文件中没有方法 {method_id!r}。"
            f"文件中已有的键为：{loaded.files}"
        )

    p0 = np.asarray(loaded[method_id], dtype=float)

    H0_RESULTS[method_id] = {
        "pvalues": p0,
        "rejection": {
            float(level): float(np.mean(p0 < float(level)))
            for level in LEVELS
        },
    }

    cutoffs = C.size_adjusted_cutoffs(
        p0,
        levels=LEVELS
    )

    print(
        f"{method_id}: "
        f"H0 reps={len(p0)}, "
        f"raw size={H0_RESULTS[method_id]['rejection']}, "
        f"adjusted cutoffs={cutoffs}"
    )

print("Loaded H0 p-values from:", H0_FILE.resolve())

In [ ]:
# Uncomment when needed.
loaded = np.load(H0_FILE)
H0_RESULTS = {}
for method_id in METHODS:
     p0 = np.asarray(loaded[method_id])
     H0_RESULTS[method_id] = {
         'pvalues': p0,
         'rejection': {
             level: float(np.mean(p0 < level))
             for level in LEVELS
         },
     }
print('Reloaded H0 calibration for:', list(H0_RESULTS))


## 7. Run H1 size-adjusted power

The key argument is:

```python
null_pvalues=H0_RESULTS[method_id]
```

The modified `ci_test.py` will raise an error if H1 is requested without matching H0 p-values. The reported values below are size-adjusted power.


In [ ]:
POWER_ROWS = []
H1_RESULTS = {method_id: {} for method_id in METHODS}

for alpha_x in ALPHA_GRID:
    print()
    print('-' * 88)
    print(f'alpha_x = {alpha_x:.2f}')
    print('-' * 88)

    for method_id, config in METHODS.items():
        h1_data_kwargs = dict(BASE_DATA_KWARGS)
        h1_data_kwargs['alpha_x'] = float(alpha_x)

        result = C.run_experiment(
            n=N,
            hypothesis='H1',
            n_rep=N_REP_H1,
            config=config,
            dgp=DGP_NAME,
            oracle=False,
            levels=LEVELS,
            data_kwargs=h1_data_kwargs,
            n_jobs=N_JOBS,
            prefer_gpu=PREFER_GPU,
            verbose=False,
            null_pvalues=H0_RESULTS[method_id],
        )

        H1_RESULTS[method_id][float(alpha_x)] = result

        row = {
            'method_id': method_id,
            'method': METHOD_LABELS[method_id],
            'alpha_x': float(alpha_x),
            'size_adjusted_power_0.10': result['size_adjusted_power'][0.10],
            'size_adjusted_power_0.05': result['size_adjusted_power'][0.05],
            'cutoff_0.10': result['cutoffs'][0.10],
            'cutoff_0.05': result['cutoffs'][0.05],
            'H1_replications': len(result['pvalues']),
        }
        POWER_ROWS.append(row)

        print(
            f"{METHOD_LABELS[method_id]:36s}  "
            f"adjusted power@0.10={row['size_adjusted_power_0.10']:.3f}  "
            f"adjusted power@0.05={row['size_adjusted_power_0.05']:.3f}"
        )

POWER_TABLE = pd.DataFrame(POWER_ROWS)
display(POWER_TABLE)


## 8. Comparison tables

These tables compare methods at the same empirical size and are the primary results to report.


In [ ]:
POWER_COMPARISON_005 = POWER_TABLE.pivot(
    index='alpha_x',
    columns='method',
    values='size_adjusted_power_0.05',
).sort_index()

POWER_COMPARISON_010 = POWER_TABLE.pivot(
    index='alpha_x',
    columns='method',
    values='size_adjusted_power_0.10',
).sort_index()

print('Size-adjusted power at empirical size 0.05')
display(POWER_COMPARISON_005)

print('Size-adjusted power at empirical size 0.10')
display(POWER_COMPARISON_010)


## 9. Power curves

In [ ]:
for level, value_column in [
    (0.05, 'size_adjusted_power_0.05'),
    (0.10, 'size_adjusted_power_0.10'),
]:
    plt.figure(figsize=(7.5, 5.0))

    for method_id in METHODS:
        subset = POWER_TABLE[
            POWER_TABLE['method_id'] == method_id
        ].sort_values('alpha_x')

        plt.plot(
            subset['alpha_x'],
            subset[value_column],
            marker='o',
            label=METHOD_LABELS[method_id],
        )

    plt.xlabel('Dependence strength alpha_x')
    plt.ylabel('Size-adjusted power')
    plt.title(f'Size-adjusted power at empirical size {level:.2f}')
    plt.ylim(0.0, 1.05)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()


## 10. Save results

In [ ]:
OUTPUT_DIR = Path(f'size_adjusted_power_results_{COMPARISON_MODE}_v2')
OUTPUT_DIR.mkdir(exist_ok=True)

POWER_TABLE.to_csv(OUTPUT_DIR / 'size_adjusted_power_long.csv', index=False)
POWER_COMPARISON_005.to_csv(OUTPUT_DIR / 'size_adjusted_power_at_0.05.csv')
POWER_COMPARISON_010.to_csv(OUTPUT_DIR / 'size_adjusted_power_at_0.10.csv')
CALIBRATION_TABLE.to_csv(OUTPUT_DIR / 'h0_calibration_summary.csv', index=False)

np.savez_compressed(
    OUTPUT_DIR / 'h0_pvalues.npz',
    **{
        method_id: np.asarray(result['pvalues'])
        for method_id, result in H0_RESULTS.items()
    },
)

h1_arrays = {}
for method_id, by_alpha in H1_RESULTS.items():
    for alpha_x, result in by_alpha.items():
        alpha_tag = f'{alpha_x:.2f}'.replace('.', 'p')
        h1_arrays[f'{method_id}_alpha_{alpha_tag}'] = np.asarray(result['pvalues'])
np.savez_compressed(OUTPUT_DIR / 'h1_pvalues.npz', **h1_arrays)

print('Saved results to:', OUTPUT_DIR.resolve())


## 11. Minimal single-method template

This is the shortest correct workflow for one method: H0 calibration first, then H1 with `null_pvalues=H0_RESULT`.


In [ ]:
# Example: only the quantile-alignment method.
ACTIVE_CONFIG = METHODS['lambda_005']

# Stage 1: matching H0 calibration
ACTIVE_H0 = C.run_experiment(
    n=N,
    hypothesis='H0',
    n_rep=N_REP_H0,
    config=ACTIVE_CONFIG,
    dgp=DGP_NAME,
    oracle=False,
    levels=LEVELS,
    data_kwargs=BASE_DATA_KWARGS,
    n_jobs=N_JOBS,
    prefer_gpu=PREFER_GPU,
    verbose=True,
)

# Stage 2: size-adjusted H1 power
ACTIVE_H1 = C.run_experiment(
    n=N,
    hypothesis='H1',
    n_rep=N_REP_H1,
    config=ACTIVE_CONFIG,
    dgp=DGP_NAME,
    oracle=False,
    levels=LEVELS,
    data_kwargs={**BASE_DATA_KWARGS, 'alpha_x': 0.20},
    n_jobs=N_JOBS,
    prefer_gpu=PREFER_GPU,
    verbose=True,
    null_pvalues=ACTIVE_H0,
)

print('Size-adjusted power:', ACTIVE_H1['size_adjusted_power'])
print('Adjusted cutoffs:', ACTIVE_H1['cutoffs'])
